# Data Preparation

This notebook constructs the final hourly dataset for electricity price forecasting
in the Netherlands (2019–2025).

Goals:
- Load all raw data sources
- Perform basic data quality checks
- Align all data to a common hourly UTC timeline
- Merge all sources into a single dataset
- Save the processed dataset for further analysis

This notebook does **not** perform exploratory analysis or time series modeling.

In [1]:
import pandas as pd
from pathlib import Path

## Project structure and paths

We define project-level paths in a centralized way to ensure reproducibility  
and avoid hard-coded file locations.

- `data/raw` contains immutable raw data as downloaded from external sources
- `data/processed` contains aligned and merged datasets used for analysis

In [2]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Helper function: loading time series data

All raw datasets come from different sources and have slightly different formats.
This helper function:
- loads a CSV file,
- parses the timestamp column,
- sets the timestamp as index,
- normalizes all timestamps to **UTC**,
- ensures correct sorting.

This guarantees consistent time handling across all data sources.

In [3]:
def load_timeseries_csv(
    path: Path,
    timestamp_col: str,
    name: str,
) -> pd.DataFrame:
    """
    Load a time series CSV file and return a DataFrame indexed by UTC timestamp.
    """
    df = pd.read_csv(path, parse_dates=[timestamp_col])
    df = df.rename(columns={timestamp_col: "timestamp"})
    df = df.set_index("timestamp").sort_index()

    # Normalize timezone
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")

    print(f"{name}: loaded {len(df)} rows")
    return df

## Helper function: temporal coverage report

To understand how well different datasets overlap in time,  
we create a summary table that shows:
- start date,
- end date,
- number of observations

for each time series.

This helps identify potential alignment issues early.

In [4]:
def coverage_report(datasets: dict) -> pd.DataFrame:
    """
    Create a summary table with date coverage information
    for multiple time series datasets.
    """
    rows = []

    for name, df in datasets.items():
        rows.append({
            "dataset": name,
            "start_date": df.index.min(),
            "end_date": df.index.max(),
            "n_rows": len(df),
        })

    return (
        pd.DataFrame(rows)
        .set_index("dataset")
        .sort_values("start_date")
    )

## Raw data sources

Below we define all raw datasets used in the project together with:
- file paths,
- timestamp column names.

This configuration-driven approach allows easy extension or replacement
of data sources without changing the core logic.

In [5]:
raw_files = {
    "prices": {
        "path": RAW_DIR / "entsoe" / "nl_day_ahead_prices_hourly_2019_2025.csv",
        "timestamp_col": "timestamp",
    },
    "load": {
        "path": RAW_DIR / "entsoe" / "nl_load_hourly_2019_2025.csv",
        "timestamp_col": "timestamp",
    },
    "solar": {
        "path": RAW_DIR / "entsoe" / "nl_solar_generation_hourly_2019_2025.csv",
        "timestamp_col": "timestamp",
    },
    "wind": {
        "path": RAW_DIR / "entsoe" / "nl_wind_generation_hourly_2019_2025.csv",
        "timestamp_col": "timestamp",
    },
    "weather": {
        "path": RAW_DIR / "weather" / "weather_hourly_nl_open_meteo_2019_2025.csv",
        "timestamp_col": "timestamp",
    },
    "gas": {
        "path": RAW_DIR / "gas" / "ttf_gas_price_daily_2019_2025.csv",
        "timestamp_col": "date",
    },
}

## Loading raw datasets

All datasets are loaded using the same helper function.
At this stage:
- no filtering is applied,
- no missing values are handled,
- no resampling is performed.

We only bring the data into a consistent in-memory representation.

In [6]:
datasets = {}

for name, cfg in raw_files.items():
    datasets[name] = load_timeseries_csv(
        path=cfg["path"],
        timestamp_col=cfg["timestamp_col"],
        name=name,
    )

prices: loaded 67989 rows
load: loaded 245472 rows
solar: loaded 245472 rows
wind: loaded 245472 rows
weather: loaded 61368 rows
gas: loaded 1761 rows


## Temporal coverage inspection

We inspect the time span of each dataset to identify:
- differences in start and end dates,
- differences in temporal resolution.

This is especially important because:
- electricity prices and generation are hourly,
- weather data may have gaps,
- gas prices are reported daily.

In [7]:
coverage_report(datasets)

,start_date,end_date,n_rows
dataset,,,
prices,2018-12-31 23:00:00+00:00,2025-12-31 23:00:00+00:00,67989
load,2018-12-31 23:00:00+00:00,2025-12-31 22:45:00+00:00,245472
solar,2018-12-31 23:00:00+00:00,2025-12-31 22:45:00+00:00,245472
wind,2018-12-31 23:00:00+00:00,2025-12-31 22:45:00+00:00,245472
weather,2019-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,61368
gas,2019-01-02 00:00:00+00:00,2025-12-30 00:00:00+00:00,1761


## Master timeline definition

The **day-ahead electricity price series** defines the master hourly timeline.

Reasons:
- it is the target variable,
- it has full hourly coverage,
- all other variables are explanatory features aligned to it.

All datasets will be reindexed to this timeline.

In [8]:
master_timeline = datasets["prices"].index

print("Master timeline:")
print(master_timeline.min(), "→", master_timeline.max())
print("Number of hours:", len(master_timeline))

Master timeline:
2018-12-31 23:00:00+00:00 → 2025-12-31 23:00:00+00:00
Number of hours: 67989


## Temporal alignment of all data sources

All datasets are reindexed to the master hourly timeline.

Notes:
- Hourly datasets are aligned directly.
- Daily gas prices are forward-filled to match hourly resolution.
- Missing values introduced by alignment are kept for now.

Handling of missing values is intentionally postponed to the analysis stage.

In [9]:
aligned = {}

aligned["prices"] = datasets["prices"].reindex(master_timeline)
aligned["load"] = datasets["load"].reindex(master_timeline)
aligned["solar"] = datasets["solar"].reindex(master_timeline)
aligned["wind"] = datasets["wind"].reindex(master_timeline)
aligned["weather"] = datasets["weather"].reindex(master_timeline)
aligned["gas"] = datasets["gas"].reindex(master_timeline, method="ffill")

## Missing values after alignment

After aligning all datasets to a common timeline,
we inspect the number of missing values per variable.

At this stage, missing values are expected due to:
- differences in reporting frequency,
- incomplete historical coverage (e.g., weather data),
- occasional missing observations in ENTSO-E data.

No imputation is performed yet.

In [10]:
for name, df in aligned.items():
    print(f"\n{name}")
    print("Missing values:")
    print(df.isna().sum())


prices
Missing values:
day_ahead_price    0
dtype: int64

load
Missing values:
load_mw    1
dtype: int64

solar
Missing values:
solar_mw    1
dtype: int64

wind
Missing values:
wind_mw    1
dtype: int64

weather
Missing values:
temperature_c    6628
wind_ms          6628
dtype: int64

gas
Missing values:
gas_price    25
dtype: int64


## Final dataset construction

All aligned datasets are merged into a single DataFrame using a left join
on the master hourly timeline.

The resulting dataset contains:
- target variable: day-ahead electricity price,
- explanatory variables:
  - electricity load,
  - solar generation,
  - wind generation,
  - weather variables,
  - gas prices.

In [11]:
final_df = (
    aligned["prices"]
    .join(aligned["load"], how="left")
    .join(aligned["solar"], how="left")
    .join(aligned["wind"], how="left")
    .join(aligned["weather"], how="left")
    .join(aligned["gas"], how="left")
)

final_df.head()

,day_ahead_price,load_mw,solar_mw,wind_mw,temperature_c,wind_ms,gas_price
timestamp,,,,,,,
2018-12-31 23:00:00+00:00,68.92,11267.78,0.22,664.76,NaN,NaN,NaN
2019-01-01 00:00:00+00:00,64.98,11265.80,0.22,659.21,7.7,17.4,NaN
2019-01-01 01:00:00+00:00,60.27,11042.94,0.22,730.78,7.7,17.7,NaN
2019-01-01 02:00:00+00:00,49.97,10802.79,0.22,776.55,7.8,19.6,NaN
2019-01-01 03:00:00+00:00,47.66,10443.23,0.22,784.86,7.8,20.8,NaN


In [12]:
final_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 67989 entries, 2018-12-31 23:00:00+00:00 to 2025-12-31 23:00:00+00:00
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   day_ahead_price  67989 non-null  float64
 1   load_mw          67988 non-null  float64
 2   solar_mw         67988 non-null  float64
 3   wind_mw          67988 non-null  float64
 4   temperature_c    61361 non-null  float64
 5   wind_ms          61361 non-null  float64
 6   gas_price        67964 non-null  float64
dtypes: float64(7)
memory usage: 4.1 MB


In [13]:
final_df.isna().sum()

day_ahead_price       0
load_mw               1
solar_mw              1
wind_mw               1
temperature_c      6628
wind_ms            6628
gas_price            25
dtype: int64

In [14]:
final_df.describe()

,day_ahead_price,load_mw,solar_mw,wind_mw,temperature_c,wind_ms,gas_price
count,67989.000000,67988.000000,67988.000000,67988.000000,61361.000000,61361.000000,67964.000000
mean,95.986665,12725.118723,42.500555,740.044858,11.266619,14.193175,43.649224
std,88.450602,2091.414175,76.401992,619.717011,6.419007,7.132405,42.684590
min,-500.000000,5349.570000,0.000000,7.120000,-12.100000,0.000000,3.510000
25%,40.700000,11112.887500,0.960000,228.010000,6.600000,8.800000,17.693750
50%,78.840000,12518.900000,1.440000,535.468500,11.100000,13.000000,31.826000
75%,115.190000,14069.775000,47.230000,1129.850000,15.900000,18.300000,44.930000
max,872.960000,19631.971000,427.720000,2546.080000,36.400000,57.800000,339.196014


## Save processed dataset

In [15]:
output_path = PROCESSED_DIR / "electricity_dataset_hourly_2019_2025.csv"
final_df.to_csv(output_path)

output_path

PosixPath('/Users/elenaibraeva/Desktop/electricity-price-forecasting-nl/data/processed/electricity_dataset_hourly_2019_2025.csv')